%md
#### Goal
- To create a dataobject to capture the staffs and coverage teams responsible for a party

#### Authors
- Mahalakshmi.Vengateswaran@rabobank.com
- Abhishek.R.Jaiswal@rabobank.com
- Prajit.tatari@rabobank.com

##### Flow of Logic
- Reading GCOB,GCDS,GIC dataobjects from GDP
- Fetching the coverage details across different dimensions from different SourceSystems
- Loading the final dataobject to SA RADAR Storage account

  ##### Version & Changes

|     Developer |Date	   | PBI/Bug No |	Changes done |
|----------|----------|----------|----------|
| Abhishek Jaiswal	 |22-July-205 |13228980 |Used explode_outer and other logic change
| Abhishek Jaiswal   |25-July-205 | 13147779| Added Legacy2 and other changes	
| Prajit Tatari   |04-Sept-205 | 13327403| Added LocationCountryCode
| Mahalakshmi V | 20-Nov-2025 | 14165735 | Made notebook compatible for Historical Load

#####READING FILES FROM GDP

In [0]:
import os
import pandas as pd
from datetime import datetime, timedelta

In [0]:
from RadarUtils import *

In [0]:
dbutils.widgets.text("Load_Date", "")
dbutils.widgets.text("RunType", "daily")  # default is daily

Load_Date = dbutils.widgets.get("Load_Date")
RunType = dbutils.widgets.get("RunType").lower()
if Load_Date:
    RunType="historical"
    
if not Load_Date:
    Load_Date = datetime.today().strftime('%Y%m%d')

print(f"Running for Load Date: {Load_Date}")

In [0]:
app_reg_app_id = os.environ["APP_REG_APP_ID"]
ReadStorage = os.environ["GDP_STORAGE_NAME"]
TenantId = os.environ["TENANT_ID"]
GIC_ReadStorage = os.environ["GDP_SA_STORAGE_NAME"]
environment = os.environ["ENV"]
radar_datamodel_version_number =3

In [0]:
party_Coverage_dataobject = "Party_Coverage"

In [0]:
SARADAR = "saradar" + environment

In [0]:
authenticate_storage_account(ReadStorage)
authenticate_storage_account(GIC_ReadStorage)
authenticate_storage_account(SARADAR)

In [0]:
# Derive the date for which data has to be processed from GDP
load_dts = f"EDL_LOAD_DTS={Load_Date}*"
print (load_dts)

In [0]:
# List of datasets from GDP
gcds_df = pd.DataFrame(
    {
        "definedDatasetname": [
            "client_Client",
            "client_KeyStoreKey",
            "client_PartyRole",
            "client_Products",
        ]
    }
)

# Create TempView for each loading table
for index, row in gcds_df.iterrows():
    Read_GDP_Defined_DataObjects(Source='GCDS', Dataobject=row.definedDatasetname,Load_Date=Load_Date)

In [0]:
# List of datasets from GDP
load_df = [
            "party_case_client_details",
            "party_AllPartyDetails",
            "party_local_client_Owners",
            "party_products_and_services",
            "party_allcountries"
        ]

# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='GCOB', Dataobject=Object, path_prefix='CaseService',Load_Date=Load_Date)

In [0]:
df_Party_SystemIdentifier = spark.read.parquet(
    f"abfss://radardatamodel@{SARADAR}.dfs.core.windows.net/Party_SystemIdentifier/3/data/{load_dts}/*.parquet"
)

In [0]:
# List of datasets from GDP
load_df = pd.DataFrame({"GDPname": [
'pessoa_linha_negocio'
,'pessoa'
,'vwgic_rdl_pessoa_linha_negocio'
,'vwgic_rdl_pessoa_tipo_cadastro'
,'pessoa_email'
]})

# Create TempView for each loading table
for index, row in load_df.iterrows():
    Read_GDP_Defined_DataObjects(Source='GIC', Dataobject=row.GDPname,Load_Date=Load_Date)

In [0]:
# List of datasets from GDP
load_df = [
'Legacy2_case_client_details'
,'Legacy2_products_and_services'
,'Legacy2_local_client_Owners'
]

# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='Legacy2', Dataobject=Object,path_prefix='Legacy2',Load_Date=Load_Date)

In [0]:
from pyspark.sql.functions import explode, array, lit

# Define the mapping between CountryISO and LocationCode
mapping = {
    'AR': ['ARG'],
    'HK': ['HKG', 'ASIA','ASRM'],
    'US': ['ATL', 'DAL', 'USANA','CHI','NARM','NEY','RAF','RDS','RNA','SAF','RSEC'],
    'AU': ['AUS', 'AUSA'],
    'CN': ['BEI', 'CHN', 'CHNDBU', 'SHA', 'SHAU'],
    'BE': ['BEL'],
    'BR': ['BRA', 'BRS', 'USASA'],
    'CA': ['CAN','CARUR'],
    'CL': ['CHL'],
    'CW': ['CUW'],
    'DE': ['DEU'],
    'ES': ['ESP'],
    'FR': ['FRA'],
    'GB': ['GBR','EURA', 'LOG', 'LONR'],
    'NL': ['GLOA', 'HOFE', 'HOFH','HOFA', 'RTN','UOTH','UPE', 'UTR', 'UTRA', 'URF', 'URFO', 'URMB','UTG','FOUND'],
    'HU': ['HUN'],
    'IE': ['IAC', 'IRL'],
    'ID': ['IDN'],
    'IN': ['IND', 'REA'],
    'IT': ['ITA'],
    'JP': ['JPN'],
    'KE': ['KEN'],
    'KR': ['KOR'],
    'MX': ['MEX', 'MEXS'],
    'MY': ['MYS'],
    'NZ': ['NZL'],
    'PE': ['PER'],
    'PL': ['POL', 'PBG'],
    'RU': ['RUS','ZAO'],
    'SG': ['SGP', 'SGPDBU'],
    'TW': ['TAI'],
    'TH': ['THA'],
    'TR': ['TUR', 'TURAS'],
    'VN': ['VNM'],
    'ZA': ['ZAF']
}

# Convert the mapping dictionary to a list of tuples
data = [(iso, loc) for iso, locs in mapping.items() for loc in locs]
# Create Spark DataFrame
df = spark.createDataFrame(data, ["CountryISO", "LocationCode"])
df.createOrReplaceTempView('ISOCode_static_table')
# display(df)

In [0]:
# Define the mapping between RegionName and LocationISOCode
mapping = {
    'Asia': ['HK','CN','ID','IN','JP','KR','MY','SG','TW','TH','TR','VN'],
    'Australia Pacific': ['AU', 'NZ'],
    'Europe': ['BE', 'DE', 'ES','GB','FR','HU','IE','IT','KE','PL','NL','RU','ZA'],
    'North America': ['US', 'CA','MX'],
    'South America': ['AR', 'BR', 'CL', 'CW', 'PE']
}

# Convert the mapping dictionary to a list of tuples
data = [(region, iso) for region, isos in mapping.items() for iso in isos]
df = spark.createDataFrame(data, ["RegionName", "LocationISOCode"])
df.createOrReplaceTempView('RegionISO_static_table')

# display(df)


#####TRANSFORMATION TO GET Party_Coverage OBJECT 

In [0]:
%sql
Create or replace temporary view Gcob_CaseClientDetails As
select
  *,
  case
    when ClientType = 'Legal Entity' then 'Legal Entity'
    when
      ClientType in ('Natural Person', 'Natural Person acting in a Professional Capacity (NPPC)')
    then
      'Natural Person'
    else null
  end as Party_type,
  Case
    when ClientType = 'Legal Entity' then concat('GCOB_LEC_', GcobId)
    when
      ClientType in ('Natural Person', 'Natural Person acting in a Professional Capacity (NPPC)')
    then
      concat('GCOB_NP_NPPC_', GcobId)
  End as LocalSystemIdentifier
from
  party_case_client_details
where
  CaseStatusName <> 'Cancelled'
  --and IsLatestApprovedVersionofclient = 'True'

In [0]:
%sql
Create or replace temporary view Gcob_AllPartyDetails_Row As
select distinct
  p.GlobalClientOwnerEmail as GlobalClientOwnerName,
  p.GlobalClientOwnerName as GCO,
  p.GlobalClientOwnerLocation,
  p.UniquePartyId,
  p.gcdsid,
  p.GcobId,
  p.PartyId,
  p.CaseId,
  case
    when p.ClientType = 'LegalEntityClient' then 'Legal Entity'
    when
      p.ClientType in (
        'NaturalPersonClient', 'Natural Person acting in a Professional Capacity (NPPC)'
      )
    then
      'Natural Person'
    when p.ClientType = 'RelatedLegalEntity' then 'Related Legal Entity'
    when p.ClientType = 'RelatedNaturalPerson' then 'Related Natural Person'
    else null
  end as Party_type,
  CONCAT('GCOB_', p.UniquePartyId) AS LocalSystemIdentifier,
  c.BusinessLineName as BusinessLine,
  lco.EmailAddress as LocalClientOwnerName,
  lco.Location as LocalClientOwnerLocation,
  pp.ProductDomain as `Product Domain`,
  pp.BusinessUnit as `Product Business Unit`,
  pp.BookingEntityLocation as BookingLocation,
  pp.ProductOfferingLocation as ProductLocation,
  ROW_NUMBER() OVER (
      PARTITION BY p.gcobid, p.ClientType
      ORDER BY
        CASE
          WHEN p.ClientLifeCycleStatus = 'Client' THEN 1
          ELSE 2
        END ASC,
        c.CaseCompletedDate DESC
    ) AS ROWNUM
from
  party_AllPartyDetails p
    Inner join Gcob_CaseClientDetails c on p.UniquePartyId = c.UniqueGcobId
      --on CASE WHEN p.ClientType = 'LegalEntityClient' THEN concat('LEC_', p.Id) WHEN p.ClientType IN (            'NaturalPersonClient', 'Natural Person acting in a Professional Capacity (NPPC)' ) THEN concat('NP_NPPC_', p.Id) end = c.SourceClient
    LEFT JOIN party_local_client_Owners lco
      on c.Sourceclient = lco.Sourceclient
    LEFT JOIN party_products_and_services pp
      on c.Sourceclient = pp.Sourceclient
where
  Status = 'Live'
  --and p.CaseStatusName <> 'Cancelled'
  --and p.IsLatestApprovedVersionofclient = 'True'

In [0]:
%sql
Create or replace temporary view Gcob_AllParty As
with gcob as 
(
select distinct
  p.GlobalClientOwnerEmail as GlobalClientOwnerName,
  p.GlobalClientOwnerName as GCO,
  p.GlobalClientOwnerLocation,
  p.UniquePartyId,
  p.gcdsid,
  p.CaseId,
  p.InternalWatchlist,
  case
    when p.ClientType = 'LegalEntityClient' then 'Legal Entity'
    when
      p.ClientType in (
        'NaturalPersonClient', 'Natural Person acting in a Professional Capacity (NPPC)'
      )
    then
      'Natural Person'
    when p.ClientType = 'RelatedLegalEntity' then 'Related Legal Entity'
    when p.ClientType = 'RelatedNaturalPerson' then 'Related Natural Person'
    else null
  end as Party_type,
  CONCAT('GCOB_', p.UniquePartyId) AS LocalSystemIdentifier,
  c.BusinessLineName as BusinessLine,
  lco.EmailAddress as LocalClientOwnerName,
  lco.Location as LocalClientOwnerLocation,
  pp.ProductDomain as `Product Domain`,
  pp.BusinessUnit as `Product Business Unit`,
  pp.BookingEntityLocation as BookingLocation,
  pp.ProductOfferingLocation as ProductLocation,
  c.SalesforceClientID_nCino,
  c.gcobid as ncino_gcobid,
  p.gcobid
  ,case when p.CaseStatusName is null then 'RelatedParty' else c.CaseStatusName end as CaseStatusName
from
  party_AllPartyDetails p
    LEFT JOIN Gcob_CaseClientDetails c on p.UniquePartyId = c.UniqueGcobId
      --on CASE WHEN p.ClientType = 'LegalEntityClient' THEN concat('LEC_', p.Id) WHEN p.ClientType IN (            'NaturalPersonClient', 'Natural Person acting in a Professional Capacity (NPPC)' ) THEN concat('NP_NPPC_', p.Id) end = c.SourceClientand p.ClientType in ('NaturalPersonClient', 'Natural Person acting in a Professional Capacity (NPPC)')
    LEFT JOIN party_local_client_Owners lco
      on c.Sourceclient = lco.Sourceclient
    LEFT JOIN party_products_and_services pp
      on c.Sourceclient = pp.Sourceclient
where
  Status = 'Live'
  --and (p.IsLatestApprovedVersionofclient = 'True' or p.IsLatestApprovedVersionofclient is null )
)
select * from gcob
where CaseStatusName <> 'Cancelled'

In [0]:
%sql
Create or replace temporary view Gcob_AllPartyDetails As
select
  *
from
  Gcob_AllPartyDetails_Row
--where
--  ROWNUM <> 2

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW NonSFDCID_Legacy2_Client AS
select * from Legacy2_case_client_details where isclient = 1 and GcobCaseId is null and Value is null and GcobId like 'RA:%';

CREATE OR REPLACE TEMPORARY VIEW Legacy2_client AS
select * from Legacy2_case_client_details
where ClientId not in (select ClientId from NonSFDCID_Legacy2_Client)
;

CREATE OR REPLACE TEMPORARY VIEW Legacy2 AS
select distinct 
c.clientId,SalesforceClientID_nCino
,CASE
  WHEN c.IsClient = 'true' and  c.ClientTypeId = 1 THEN concat('LE_', GcobId)
  WHEN c.IsClient = 'true' and c.ClientTypeId in (2,3) THEN concat('NP_NPPC_', GcobId)
  WHEN c.IsClient = 'false' and c.ClientTypeId = 1 THEN concat('RLE_', GcobId)
  WHEN c.IsClient = 'false' and c.ClientTypeId in (2,3) THEN concat('RNP_', GcobId)
END as Legacy2_Identifier
,case when c.ClientType = 'Legal Entity' then 'Legal Entity' 
      when c.ClientType in ('Natural Person','Natural Person acting in a Professional Capacity (NPPC)') then 'Natural Person' 
      when c.ClientType = 'Related Legal Entity' then 'Related Legal Entity' 
      when c.ClientType = 'Related Natural Person' then 'Related Natural Person' 
      else null
end as Party_type
,concat('LEGACY2_',Legacy2_Identifier) as LocalSystemIdentifier
,c.GlobalClientOwner as GCO
,c.GlobalClientOwnerEmail as GlobalClientOwnerName
,c.GlobalClientOwnerLocation
,c.BusinessLineName as BusinessLine
,lco.LocalOwnerEmailAddress as LocalClientOwnerName
,lco.LocalClientOwnerCountry as LocalClientOwnerLocation
,null as `Product Domain`
,null as `Product Business Unit`
,pp.BookingLocation as BookingLocation
,pp.ProductLocation as ProductLocation
from Legacy2_client c
LEFT JOIN Legacy2_local_client_Owners lco on c.ClientId = lco.ClientId
    LEFT JOIN Legacy2_products_and_services pp on c.ClientId = pp.ClientId
where ClientTypeId in (1,2,3)
--and IsLatestApprovedVersionofclient = 'True'

In [0]:
%sql
Create or replace temporary view GIC_ClientDetails As
With BusinessLine as
(
select distinct 
vng.COD_INSTITUCIONAL
,vng.DES_LINHA_NEGOCIO as BusinessLine
From vwgic_rdl_pessoa_linha_negocio vng 
LEFT JOIN vwgic_rdl_pessoa_tipo_cadastro cad on cad.SEQ_PESSOA = vng.SEQ_PESSOA 
where vng.DTA_DESATIVACAO is null
and cad.SEQ_TIPO_CADASTRO = 1 
and SEQ_STATUS_TIPO_CADASTRO in (1,3)
)
select distinct
  p.COD_INSTITUCIONAL,
  CASE
    WHEN p.SEQ_TIPO_PESSOA = '2' THEN 'Natural Person'
    else 'Legal Entity'
  END AS Party_type,
  CONCAT('GIC_', p.COD_INSTITUCIONAL) as LocalSystemIdentifier
  ,'Rabobank Brazil' as GlobalClientOwnerLocation
  ,gco.NOM_COMPLETO as GCO
  ,e.DES_EMAIL as GlobalClientOwnerName
  ,b.BusinessLine
from
  pessoa p
  LEFT JOIN PESSOA_LINHA_NEGOCIO ng on p.SEQ_PESSOA = ng.SEQ_PESSOA
  LEFT JOIN pessoa gco  on ng.SEQ_PESSOA_RM = gco.SEQ_PESSOA
  LEFT JOIN pessoa_email e on gco.SEQ_PESSOA = e.SEQ_PESSOA
  LEFT JOIN BusinessLine b on p.COD_INSTITUCIONAL = b.COD_INSTITUCIONAL 

In [0]:
%sql
Create or replace temporary view GCDS_Clients As
select distinct
  k.KeyStore_value as identifier,
  k.KeyStore_type,
  c.Party_Type,
  pr.Party_role,
  c.GCID,
  c.`Global_CO-email` AS GlobalClientOwnerName,
  c.`Global_CO-name` as GCO,
  c.`CO-email` as LocalClientOwnerName,
  `Global_CO-businessline_description`,
  `Global_CO-businessline_code`,
  c.`Global_CO-location`,
  c.`CO-location`,
  P.BOOKING_LOCATION,
  P.PRODUCT_LOCATION,
  null as `Product Domain`,
  P.BusinessUnit as `Product Business Unit`,
  c.`Global_CO-location_code`
from
  client_KeyStoreKey k
    inner join client_Client c
      on c.GCID = k.GCID
    left join client_PartyRole pr
      on k.GCID = pr.GCID
    left join client_Products p
      on p.GCID = c.GCID
--where
--  KeyStore_type in ('GCOBID', 'NCINOID', 'GIC', 'CB RANZ')
  --and k.KeyStore_value not in (select distinct gcobid from Gcob_AllPartyDetails_Row where ROWNUM = 2 and gcdsid is not null )

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW Coverage_GCDS AS
WITH BaseCoverage AS (
SELECT
  CASE WHEN gcds.identifier = gcob.GcobId AND gcob.Party_type = 'Legal Entity' AND gcds.KeyStore_type = 'GCOBID'
       THEN gcob.LocalSystemIdentifier
       --WHEN c.SalesforceClientID_nCino = gcds.identifier AND gcds.KeyStore_type = 'NCINOID' THEN c.LocalSystemIdentifier
       WHEN c.ClientId = ncino.PartyId AND ncino.Party_type = 'Natural Person'
       THEN ncino.LocalSystemIdentifier
       WHEN gcds.identifier = gic.COD_INSTITUCIONAL AND gcds.KeyStore_type = 'GIC'
       THEN gic.LocalSystemIdentifier
       WHEN l2.SalesforceClientID_nCino = gcds.identifier AND l2.Party_type = gcds.Party_type AND gcds.KeyStore_type = 'NCINOID'
       THEN l2.LocalSystemIdentifier
       else CONCAT('GCDS_', gcds.GCID)
  END AS LocalSystemIdentifier,
  CASE WHEN gcds.identifier = gcob.GcobId AND gcob.Party_type = 'Legal Entity' AND gcds.KeyStore_type = 'GCOBID'
       THEN 'GCOB'
       --WHEN c.SalesforceClientID_nCino = gcds.identifier AND gcds.KeyStore_type = 'NCINOID' THEN 'GCOB'
       WHEN c.ClientId = ncino.PartyId AND ncino.Party_type = 'Natural Person'
       THEN 'GCOB'
       WHEN gcds.identifier = gic.COD_INSTITUCIONAL AND gcds.KeyStore_type = 'GIC'
       THEN 'GIC'
       WHEN l2.SalesforceClientID_nCino = gcds.identifier AND l2.Party_type = gcds.Party_type AND gcds.KeyStore_type = 'NCINOID'
       THEN 'Legacy2'
       Else 'GCDS'
  END AS Application,
  --COALESCE(gcob.GlobalClientOwnerName, ncino.GlobalClientOwnerName, gic.GlobalClientOwner, gcds.GlobalClientOwnerName) AS GlobalClientOwnerName,
    array_distinct(
      filter(
        array(gcob.LocalClientOwnerName, ncino.LocalClientOwnerName, gic.GlobalClientOwnerName, l2.LocalClientOwnerName, gcds.LocalClientOwnerName),
        x -> x IS NOT NULL
      )
    ) AS LocalClientOwnerNameArray,
  --COALESCE(gcds.`Global_CO-businessline_description`, gcob.BusinessLine, ncino.BusinessLine) AS BusinessLine,
  --COALESCE(gcob.GlobalClientOwnerLocation, ncino.GlobalClientOwnerLocation, gic.GlobalClientOwnerLocation, gcds.`Global_CO-location`) AS GlobalClientOwnerLocation,
    array_distinct(
      filter(
        array(gcob.LocalClientOwnerLocation, ncino.LocalClientOwnerLocation, gic.GlobalClientOwnerLocation,l2.LocalClientOwnerLocation, gcds.`CO-location`),
        x -> x IS NOT NULL
      )
    ) AS LocalClientOwnerLocationArray,
    array_distinct(
      filter(
        array(gcds.BOOKING_LOCATION, gcob.BookingLocation, ncino.BookingLocation, l2.BookingLocation),
        x -> x IS NOT NULL
      )
    ) AS BookingLocationArray,
    array_distinct(
      filter(
        array(gcds.PRODUCT_LOCATION, gcob.ProductLocation, ncino.ProductLocation, l2.ProductLocation),
        x -> x IS NOT NULL
      )
    ) AS ProductLocationArray,
    COALESCE(
      gcds.`Product Business Unit`, gcob.`Product Business Unit`, ncino.`Product Business Unit`
    ) AS `Product Business Unit`,
    COALESCE(gcob.`Product Domain`, ncino.`Product Domain`) AS `Product Domain`
    --,gcds.GCID
    ,COALESCE(gcob.GCO, ncino.GCO, gic.GCO, l2.GCO, gcds.GCO) AS GCO
    ,COALESCE(gcob.GlobalClientOwnerName, ncino.GlobalClientOwnerName, gic.GlobalClientOwnerName, l2.GlobalClientOwnerName, gcds.GlobalClientOwnerName) AS GlobalClientOwnerName
    ,COALESCE(gcds.`Global_CO-businessline_description`, gcob.BusinessLine, ncino.BusinessLine,gic.BusinessLine) AS BusinessLine
    ,COALESCE(gcob.GlobalClientOwnerLocation, ncino.GlobalClientOwnerLocation, gic.GlobalClientOwnerLocation, l2.GlobalClientOwnerLocation, gcds.`Global_CO-location`) AS GlobalClientOwnerLocation,
    gcds.`Global_CO-location_code` as Location_code
  FROM
    GCDS_cLIENTS gcds
      LEFT JOIN Gcob_AllPartyDetails gcob
        ON gcds.identifier = gcob.GcobId
        --AND gcds.Party_type = gcob.Party_type
        --AND gcds.GCID = gcob.GCDSID
        AND gcob.Party_type = 'Legal Entity'
        --AND gcds.party_role = 'Customer'
        AND gcds.KeyStore_type = 'GCOBID'
      LEFT JOIN Gcob_CaseClientDetails c
        ON c.SalesforceClientID_nCino = gcds.identifier
        --AND c.Party_type = gcds.Party_type
        AND gcds.KeyStore_type = 'NCINOID'
        --AND gcds.party_role = 'Customer'
      LEFT JOIN Gcob_AllPartyDetails ncino
        ON c.ClientId = ncino.PartyId
        AND ncino.Party_type = 'Natural Person'
        --AND gcds.party_role = 'Customer'
      LEFT JOIN GIC_ClientDetails gic
        ON gcds.identifier = gic.COD_INSTITUCIONAL
        AND gcds.KeyStore_type = 'GIC'
        --AND gcds.party_role = 'Customer'
      LEFT JOIN Legacy2 l2 on l2.SalesforceClientID_nCino = gcds.identifier 
        and l2.Party_type = gcds.Party_type
        and gcds.KeyStore_type = 'NCINOID'
        --and gcds.party_role = 'Customer' 
)
SELECT distinct
  LocalSystemIdentifier,
  Application,
  GCO,
  GlobalClientOwnerName,
  explode_outer(LocalClientOwnerNameArray) AS LocalClientOwnerName,
  BusinessLine,
  GlobalClientOwnerLocation,
  explode_outer(LocalClientOwnerLocationArray) AS LocalClientOwnerLocation,
  explode_outer(BookingLocationArray) AS `Booking Location`,
  explode_outer(ProductLocationArray) AS `Product Location`,
  `Product Business Unit`,
  `Product Domain`,
  Location_code
  --,GCID
FROM
  BaseCoverage

In [0]:
%sql
Create or replace temporary view GCOB_Coverage_NonGCDS As
with GCOB as (
  select distinct
    t1.LocalSystemIdentifier,
    'GCOB' as Application,
    t1.Party_Type,
    t1.CaseId,
    t1.InternalWatchlist,
    t1.GCO,
    t1.GlobalClientOwnerName,
    t1.LocalClientOwnerName,
    t1.BusinessLine,
    t1.GlobalClientOwnerLocation,
    t1.LocalClientOwnerLocation,
    t1.BookingLocation,
    t1.ProductLocation,
    t1.`Product Business Unit`,
    t1.`Product Domain`
  from
    Gcob_AllParty t1
  where
    t1.Party_Type <> 'Natural Person'
  union
  select distinct
    t1.LocalSystemIdentifier,
    'GCOB' as Application,
    t1.Party_Type,
    t1.CaseId,
    t1.InternalWatchlist,
    t1.GCO,
    t1.GlobalClientOwnerName,
    t1.LocalClientOwnerName,
    t1.BusinessLine,
    t1.GlobalClientOwnerLocation,
    t1.LocalClientOwnerLocation,
    t1.BookingLocation,
    t1.ProductLocation,
    t1.`Product Business Unit`,
    t1.`Product Domain`
  from
    Gcob_AllParty t1
  where
    t1.Party_Type = 'Natural Person'
)
select distinct
  LocalSystemIdentifier,
  Application,
  GCO,
  GlobalClientOwnerName,
  LocalClientOwnerName,
  BusinessLine,
  GlobalClientOwnerLocation,
  LocalClientOwnerLocation,
  BookingLocation,
  ProductLocation,
  `Product Business Unit`,
  `Product Domain`,
  ROW_NUMBER() OVER (
      PARTITION BY LocalSystemIdentifier, Party_Type
      ORDER BY caseid desc, InternalWatchlist desc
    ) AS ROWNUM
from
  GCOB

In [0]:
%sql
Create or replace temporary view Coverage_NonGCDS As
with NonGCDS as (
  select
    *
  from
    GCOB_Coverage_NonGCDS
  union
  select distinct
    t1.LocalSystemIdentifier,
    'GIC' as Application,
    t1.GCO,
    t1.GlobalClientOwnerName as GlobalClientOwnerName,
    t1.GlobalClientOwnerName as LocalClientOwnerName,
    null as BusinessLine,
    t1.GlobalClientOwnerLocation as GlobalClientOwnerLocation,
    t1.GlobalClientOwnerLocation as LocalClientOwnerLocation,
    null as BookingLocation,
    null as ProductLocation,
    null as Product_Business_Unit,
    null as Product_Domain,
    ROW_NUMBER() OVER (
        PARTITION BY t1.COD_INSTITUCIONAL, t1.Party_Type
        ORDER BY t1.COD_INSTITUCIONAL DESC
      ) AS ROWNUM
  From
    GIC_ClientDetails t1

union
  select distinct
    t1.LocalSystemIdentifier,
    'Legacy2' as Application,
    t1.GCO,
    t1.GlobalClientOwnerName as GlobalClientOwnerName,
    t1.LocalClientOwnerName as LocalClientOwnerName,
    t1.BusinessLine as BusinessLine,
    t1.GlobalClientOwnerLocation as GlobalClientOwnerLocation,
    t1.LocalClientOwnerLocation as LocalClientOwnerLocation,
    t1.BookingLocation as BookingLocation,
    t1.ProductLocation as ProductLocation,
    null as Product_Business_Unit,
    null as Product_Domain,
    ROW_NUMBER() OVER (PARTITION BY t1.Legacy2_Identifier ORDER BY t1.clientid DESC) AS ROWNUM
  From
    Legacy2 t1
)
select distinct
  LocalSystemIdentifier,
  Application,
  GCO,
  GlobalClientOwnerName,
  LocalClientOwnerName,
  BusinessLine,
  GlobalClientOwnerLocation,
  LocalClientOwnerLocation,
  BookingLocation,
  ProductLocation,
  `Product Business Unit`,
  `Product Domain`,
  null as Location_code
from
  NonGCDS as t1
where
  t1.ROWNUM = 1

In [0]:
%sql
Create or replace temporary view NonGCDS_UniqueCoverage As
select * 
from Coverage_NonGCDS a
left anti join Coverage_GCDS b
on a.LocalSystemIdentifier = b.LocalSystemIdentifier


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW Coverage AS
SELECT * FROM Coverage_GCDS WHERE Application IS NOT NULL
UNION
--SELECT * FROM Coverage_NonGCDS WHERE Localsystemidentifier NOT IN (SELECT Localsystemidentifier FROM Coverage_GCDS );
select * from NonGCDS_UniqueCoverage

In [0]:
df_party_Coverage = spark.table("Coverage")

In [0]:
df_f_party_Coverage = add_party_identifier(df_party_Coverage, df_Party_SystemIdentifier)

In [0]:
df_f_party_Coverage.createOrReplaceTempView('Party_Coverage')

In [0]:
%sql
Create or replace temporary view SystemPrefAttribute As
WITH ranked_data AS (
SELECT distinct
PartyIdentifier,
GCO,
ROW_NUMBER() OVER (PARTITION BY PartyIdentifier ORDER BY
  CASE
  WHEN GCO IS NOT NULL AND Application = 'GCOB' THEN 1
  WHEN GCO IS NOT NULL AND Application = 'Legacy2' THEN 2
  WHEN GCO IS NOT NULL AND Application = 'GIC' THEN 3
  WHEN GCO IS NOT NULL AND Application = 'GCDS' THEN 4
  ELSE 10 END 
) AS rn_GCO,
GlobalClientOwnerLocation,
ROW_NUMBER() OVER (PARTITION BY PartyIdentifier ORDER BY
  CASE
  WHEN GlobalClientOwnerLocation IS NOT NULL AND Application = 'GCOB' THEN 1
  WHEN GlobalClientOwnerLocation IS NOT NULL AND Application = 'Legacy2' THEN 2
  WHEN GlobalClientOwnerLocation IS NOT NULL AND Application = 'GIC' THEN 3
  WHEN GlobalClientOwnerLocation IS NOT NULL AND Application = 'GCDS' THEN 4
  ELSE 10 END
) AS rn_GlobalClientOwnerLocation,
BusinessLine,
ROW_NUMBER() OVER (PARTITION BY PartyIdentifier ORDER BY
  CASE
  WHEN BusinessLine IS NOT NULL AND Application = 'GCOB' THEN 1
  WHEN BusinessLine IS NOT NULL AND Application = 'Legacy2' THEN 2
  WHEN BusinessLine IS NOT NULL AND Application = 'GIC' THEN 3
  WHEN BusinessLine IS NOT NULL AND Application = 'GCDS' THEN 4
  ELSE 10 END
) AS rn_BusinessLine,
Application
FROM Party_Coverage
)

select distinct 
PartyIdentifier
,MAX(CASE WHEN rn_GCO = 1 THEN GCO END) AS GCO
,MAX(CASE WHEN rn_GlobalClientOwnerLocation = 1 THEN GlobalClientOwnerLocation END) AS GlobalClientOwnerLocation
,MAX(CASE WHEN rn_BusinessLine = 1 THEN BusinessLine END) AS BusinessLine
from ranked_data
where PartyIdentifier is not null
group by PartyIdentifier

In [0]:
%sql
Create or replace temporary view GCO_Email As
select distinct
c.PartyIdentifier,
c.Application,
a.GCO,
c.GlobalClientOwnerName
from Party_Coverage c
Inner join SystemPrefAttribute a on c.PartyIdentifier = a.PartyIdentifier and c.GCO = a.GCO
where c.GlobalClientOwnerName is not null

In [0]:
%sql
Create or replace temporary view P_Coverage As

with all_countries as (
  select distinct split(Name, ' \\(the\\)')[0] as Name, ISOCode from party_allcountries
order by Name
),
GCDS_countries as (
  select distinct `Global_CO-location` AS GCDS_Location,`Global_CO-location_code` AS LocationCode,i.CountryISO as LocationISOCode
from client_Client cc
left join ISOCode_static_table i on i.LocationCode = cc.`Global_CO-location_code`
where `Global_CO-location_code` is not null
qualify row_number() over(partition by `Global_CO-location` order by `Global_CO-location_code` desc) = 1
)
select distinct
c.PartyIdentifier,
c.Application,
e.GlobalClientOwnerName,
c.LocalClientOwnerName,
a.BusinessLine,
a.GlobalClientOwnerLocation,
coalesce(ac1.ISOCode, ctr1.LocationISOCode) AS GCO_Location_ISOCode,
c.LocalClientOwnerLocation,
coalesce(ac2.ISOCode, ctr2.LocationISOCode) AS LCO_Location_ISOCode,
c.`Booking Location`,
ac3.ISOCode AS Booking_Location_ISOCode,
c.`Product Location`,
ac4.ISOCode AS Product_Location_ISOCode,
c.`Product Business Unit`,
c.`Product Domain`,
c.Location_code as LocationCode
from Party_Coverage c
left join GCO_Email e on c.PartyIdentifier = e.PartyIdentifier
left join SystemPrefAttribute a on c.PartyIdentifier = a.PartyIdentifier
left join all_countries ac1 on a.GlobalClientOwnerLocation = ac1.Name
left join all_countries ac2 on c.LocalClientOwnerLocation = ac2.Name
left join all_countries ac3 on c.`Booking Location` = ac3.Name
left join all_countries ac4 on c.`Product Location` = ac4.Name
left join GCDS_countries ctr1 on a.GlobalClientOwnerLocation = ctr1.GCDS_Location
left join GCDS_countries ctr2 on c.LocalClientOwnerLocation = ctr2.GCDS_Location
--where c.PartyIdentifier = 'GCDS_2723942'

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW InvolvedLocation AS
select distinct
  PartyIdentifier,
  Application,
  'Location' as CoverageType,
  'Involved Location' as CoverageTypeDescription,
  `Product Location` as CoverageValue,
  Product_Location_ISOCode AS LocationCountryCode,
  null as LocationCode
from
  P_Coverage
Union
select distinct
  PartyIdentifier,
  Application,
  'Location' as CoverageType,
  'Involved Location' as CoverageTypeDescription,
  `Booking Location` as CoverageValue,
  Booking_Location_ISOCode AS LocationCountryCode,
  null as LocationCode
from
  P_Coverage

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW Coverages AS
select distinct
  PartyIdentifier,
  Application,
  'GCO' as CoverageType,
  'Global Client Owner' as CoverageTypeDescription,
  GlobalClientOwnerName as CoverageValue,
  NULL AS LocationCountryCode,
  null as LocationCode
from
  P_Coverage
UNION
select distinct
  PartyIdentifier,
  Application,
  'LCO' as CoverageType,
  'Local Client Owner' as CoverageTypeDescription,
  LocalClientOwnerName as CoverageValue,
  NULL AS LocationCountryCode,
  null as LocationCode
from
  P_Coverage
UNION
select distinct
  PartyIdentifier,
  Application,
  'Business Line' as CoverageType,
  '' as CoverageTypeDescription,
  BusinessLine as CoverageValue,
  NULL AS LocationCountryCode,
  null as LocationCode
from
  P_Coverage
UNION
select distinct
  PartyIdentifier,
  Application,
  'Location' as CoverageType,
  'GCO Location' as CoverageTypeDescription,
  GlobalClientOwnerLocation as CoverageValue,
  GCO_Location_ISOCode AS LocationCountryCode,
  LocationCode
from
  P_Coverage
UNION
select distinct
  PartyIdentifier,
  Application,
  'Location' as CoverageType,
  'LCO Location' as CoverageTypeDescription,
  LocalClientOwnerLocation as CoverageValue,
  LCO_Location_ISOCode AS LocationCountryCode,
  LocationCode
from
  P_Coverage
UNION
select distinct
  PartyIdentifier,
  Application,
  'Product Business Unit' as CoverageType,
  '' as CoverageTypeDescription,
  `Product Business Unit` as CoverageValue,
  NULL AS LocationCountryCode,
  null as LocationCode
from
  P_Coverage
Union
select distinct
  PartyIdentifier,
  Application,
  'Product Domain' as CoverageType,
  '' as CoverageTypeDescription,
  `Product Domain` as CoverageValue,
  NULL AS LocationCountryCode,
  null as LocationCode
from
  P_Coverage
UNION
SELECT DISTINCT
  PartyIdentifier,
  Application,
  CoverageType,
  CoverageTypeDescription,
  CoverageValue,
  LocationCountryCode,
  LocationCode
FROM
  InvolvedLocation

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW Coverage_Final AS
select c.*,s.RegionName
from Coverages c
left join RegionISO_static_table s on c.LocationCountryCode = s.LocationISOCode

##### CREATING A FINAL DATAFRAME BY COMBINING ALL THE INFORMATION FROM DIFFERENT SOURCESYSTEMS AND LOADING THE FINAL DATAOBJECT TO SA RADAR STORAGE ACCOUNT

In [0]:
df_final_party_Coverage = spark.table("Coverage_Final")

In [0]:
if RunType == "historical":
    save_to_saradar_storage_account(df_final_party_Coverage, party_Coverage_dataobject, radar_datamodel_version_number, environment,Load_Date)
else:
    save_to_saradar_storage_account(df_final_party_Coverage, party_Coverage_dataobject, radar_datamodel_version_number, environment)